# This is a public coursework notebook. The data is an open benchmark dataset
>
> Nothing in this notebook is taken from, derived from, or capable of revealing any confidential client or institutional transaction data. No real customer identifier, card number, or personally identifiable information appears in the working extract.
>
> The analysis uses the publicly available ULB Credit Card Fraud dataset (Kaggle / DbCL-1.0). The features are largely PCA-transformed anonymised variables supplied with the benchmark. The notebook exists to study *unsupervised anomaly detection* under severe class imbalance; it is not intended to represent any specific bank's production fraud system.

---


# Detecting fraudulent card transactions with unsupervised anomaly detection

### CAM DS 2025 | Data Science with ML and AI | University of Cambridge PACE

---

## Project Overview

This notebook builds an unsupervised anomaly-detection pipeline for credit-card fraud using the ULB Credit Card Fraud benchmark. Labels are treated as incomplete or noisy for *training*: models learn what normal transactions look like and flag deviations for human review. Held-out labels are used only to evaluate how well those flags recover known fraud.

According to [UK Finance (2024)](https://www.ukfinance.org.uk/system/files/2024-06/UK%20Finance%20Annual%20Fraud%20report%202024.pdf), **fraud losses in the UK exceeded £1.2B**, with authorised push payment (APP) and card fraud accounting for a large share of cases. Globally, financial institutions lose an estimated $400B+ annually to fraudulent transactions. Early detection therefore carries both financial and reputational weight.

A central cost asymmetry shapes the modelling choices that follow. **A false negative (missed fraud) costs much more than a false positive (a legitimate transaction sent to manual review).** The pipeline therefore biases towards higher recall, accepting a controllable volume of false alerts, while keeping alert rates operationally manageable.

Anomaly detection can also amplify data bias: under-represented but legitimate segments may look rare and be over-flagged. Under GDPR and the UK Equality Act, automated decisions that systematically disadvantage a group can be unlawful. Reviewers need explanatory tools, and any personally identifiable information must remain anonymised and used only for fraud prevention.

* **Data source:** ULB Credit Card Fraud dataset via Kaggle (`mlg-ulb/creditcardfraud`); public benchmark under DbCL-1.0
* **Domain:** card-transaction fraud detection; unsupervised anomaly detection under severe class imbalance
* **EDA focus:** missingness and duplicates, class imbalance, feature distributions, correlation and leakage checks, temporal sanity
* **Primary models and metric:** Isolation Forest, Local Outlier Factor (LOF) and One-Class SVM; evaluation prioritises recall with precision–recall threshold analysis

---



## Research Questions

This analysis is structured around five questions:

1. **What does "normal" look like in this transaction panel, and how separable are known fraud cases in feature space?**
  Given PCA-transformed features plus `Time` and `amount`, which exploratory patterns (imbalance, skew, correlation) set realistic expectations for unsupervised detectors?

2. **Given that false negatives cost far more than false positives, how should evaluation and thresholds be set?**
  How should contamination, score ranking and precision–recall analysis be used so that recall is prioritised without producing an unmanageable alert volume?

3. **Which unsupervised method best recovers fraud under that cost trade-off?**
  Do Isolation Forest, Local Outlier Factor and One-Class SVM differ materially in recall, precision and operational alert rate when judged on the same protocol?

4. **Are the supplied PCA components plus scaled `Time` and `amount` sufficient as inputs, or does feature preparation change outcomes?**
  How should scaling and feature choices be documented so that models remain fair to compare and free of label leakage into training?

5. **What robustness, fairness and governance constraints should bound automated flagging?**
  Where might rare but legitimate behaviour be over-flagged, what explanations do reviewers need, and how do GDPR and institutional ethics limit deployment?

---

## Analytical Plan

The notebook follows a structured unsupervised fraud-detection pipeline. Stages 1 to 3 are implemented here as environment setup, data loading and initial exploratory checks. Stages 4 to 12 are deferred and are not implemented in this notebook.

**STAGE 1: Environment and package setup**
Import libraries, set display options and a fixed random seed, and define column-name constants used throughout the notebook.

**STAGE 2: Data loading and description**
Fetch the ULB Credit Card Fraud extract via the repository Kaggle helper, load it into a DataFrame, standardise key column names, and report shape and fraud rate.

**STAGE 3: Exploratory data analysis**
Assess missingness, duplicates, class imbalance, feature distributions, correlation and leakage indicators, and temporal sanity checks that may affect modelling.

**STAGE 4: Feature preparation**
Prepare unsupervised-ready inputs by retaining the PCA components and scaling `Time` and `amount`, documenting choices that avoid leaking the label into training features.

**STAGE 5: Evaluation protocol**
Define how labels are used only for evaluation, including contamination assumptions, anomaly-score ranking and any hold-out arrangement that keeps training unsupervised.

**STAGE 6: Baseline anomaly detector**
Fit a simple Isolation Forest baseline on normal-pattern features and record recall-oriented reference metrics against the evaluation labels.

**STAGE 7: Model comparison**
Compare Isolation Forest, Local Outlier Factor and One-Class SVM under the same protocol, judging each on recall, precision and alert volume.

**STAGE 8: Threshold selection**
Use precision–recall analysis to choose an operating threshold that deliberately favours recall while keeping false alarms reviewable.

**STAGE 9: Explainability**
Provide feature-attribution or similar explanations for flagged cases so manual reviewers can interpret why an alert was raised.

**STAGE 10: Robustness, ethics and risks**
Stress-test stability, document bias and fairness risks, and record GDPR-aligned constraints on automated decisions and PII use.

**STAGE 11: Conclusions and recommendations**
Summarise the strongest unsupervised approach, the cost–threshold trade-off, and practical recommendations for review workflows.

**STAGE 12: Reproducibility appendix**
Record package versions and export a requirements snapshot so the analysis can be re-run end to end.



---

## 1. STAGE 1: Environment and package setup

Import libraries, set display options and a fixed random seed, and define column-name constants used throughout the notebook.



### 1.1. Import required packages

Import the libraries needed for data loading, cleaning, visualisation and later modelling, and fix the random seed for reproducibility.



In [ ]:
import gc
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("error")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

SEED = 79
LABEL_COL = "label"
AMOUNT_COL = "amount"
TIME_COL = "Time"

np.random.seed(SEED)
plt.style.use("default")

print("Python", sys.version)
print("pandas", pd.__version__)


def release_memory():
    gc.collect()


---

## 2. STAGE 2: Data loading and description

Fetch the ULB Credit Card Fraud extract via the repository Kaggle helper, load it into a DataFrame, standardise key column names, and report shape and fraud rate.



### 2.1. Fetch and load the ULB extract

Download the public Kaggle dataset with `scripts/kaggle_fetch.py` (requires `kaggle.json` in the repository `.kaggle` folder), then load `creditcard.csv` and rename `Class` / `Amount` for consistent use downstream.



In [ ]:
!python ../scripts/kaggle_fetch.py mlg-ulb/creditcardfraud creditcard.csv data/ulb
df = pd.read_csv('../data/ulb/creditcard.csv')
df.rename(columns={'Class':'label','Amount':'amount'}, inplace=True)
print("ULB shape:", df.shape); display(df.head())
print("Fraud rate:", df['label'].mean())

---

## 3. STAGE 3: Exploratory data analysis

Assess missingness, duplicates, class imbalance, feature distributions, correlation and leakage indicators, and temporal sanity checks that may affect modelling.

Having confirmed the dataset shape and overall fraud rate, the next checks establish feature quality and relationships before any unsupervised modelling. The exploratory agenda is:

1. Missingness and duplicates in features.
2. Class imbalance (visual check).
3. Feature distributions and skew.
4. Correlation and leakage indicators.
5. Temporal sanity checks where applicable.



### 3.1. Are any values missing?

A missing value is simply a blank: we do not know that fact about the transaction. If lots of blanks appear in an important column, later analysis can be misleading. The check below counts blanks in every column, including the fraud label.


In [ ]:
blank_counts = df.isna().sum()
blank_report = pd.DataFrame({
    "blanks": blank_counts,
    "pct_blank": df.isna().mean().mul(100),
}).sort_values("blanks", ascending=False, kind="stable")

display(blank_report.round({"pct_blank": 3}))
print(f"Total blank cells in the whole table: {blank_counts.sum():,}")

### 3.2. Are any rows repeated?

A duplicate is the same row appearing more than once. That can happen if a transaction was saved twice. Counting repeats helps us spot that problem early. We do **not** delete anything yet — we only look at the numbers and decide later.


In [ ]:
extra_copies = int(df.duplicated().sum())
rows_in_repeat_groups = int(df.duplicated(keep=False).sum())

print(f"Total rows: {len(df):,}")
print(f"Extra copies of an identical row: {extra_copies:,}")
print(f"Rows that belong to a repeated group (original + copies): {rows_in_repeat_groups:,}")
print("No rows were removed; this is a count only.")

### 3.3. How rare is fraud?

Fraud should be uncommon. If almost every row is legitimate, a model that always says "normal" looks accurate but misses the rare cases that matter. The chart and counts below show how uneven the two classes are.



In [ ]:
class_counts = df[LABEL_COL].value_counts().sort_index()
class_counts.index = ["Legitimate (0)", "Fraud (1)"]
class_pct = class_counts.div(len(df)).mul(100)

summary = pd.DataFrame({"count": class_counts, "percent": class_pct.round(3)})
display(summary)

print(
    f"Out of {len(df):,} transactions, "
    f"{int(class_counts['Fraud (1)']):,} are fraud "
    f"({class_pct['Fraud (1)']:.3f}%)."
)

### 3.4. What do transaction amounts look like?

`amount` is the spend size of each transaction — the easiest column to read without specialist knowledge. Comparing legitimate vs fraud amounts shows whether fraud tends to be small, large, or mixed.

In [ ]:
amount_by_class = (
    df.groupby(LABEL_COL)[AMOUNT_COL]
    .describe()
    .rename(index={0: "Legitimate", 1: "Fraud"})
)
display(amount_by_class.round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, label_value, title, color in zip(
    axes,
    [0, 1],
    ["Legitimate amounts", "Fraud amounts"],
    ["steelblue", "darkorange"],
):
    values = df.loc[df[LABEL_COL] == label_value, AMOUNT_COL]
    ax.hist(values, bins=40, color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Amount")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

leg_med = amount_by_class.loc["Legitimate", "50%"]
fraud_med = amount_by_class.loc["Fraud", "50%"]
leg_mean = amount_by_class.loc["Legitimate", "mean"]
fraud_mean = amount_by_class.loc["Fraud", "mean"]

print(
    f"Median amount — legitimate: {leg_med:.2f}, fraud: {fraud_med:.2f}. "
    f"Mean amount — legitimate: {leg_mean:.2f}, fraud: {fraud_mean:.2f}."
)

### 3.4. Analysis: Fraud amounts

Fraud amounts are **mixed**. The typical (median) fraud is *smaller* than the typical legitimate payment, but the fraud *average* is higher because a minority of frauds are larger. There is no single “fraud is always small” or “always large” rule.

### 3.6. Does fraud look different over time?

`Time` is seconds since the first transaction in the file. The extract covers about **two days**. We treat the start as Day 1 at 00:00 (the true calendar clock is unknown) and plot each day by **hour of day** (00:00–23:00), so patterns are easier to read.

In [ ]:
eda_time = df[[TIME_COL, LABEL_COL]].copy()
elapsed_hour = (eda_time[TIME_COL] // 3600).astype(int)
eda_time["day"] = (elapsed_hour // 24) + 1
eda_time["hour_of_day"] = elapsed_hour % 24

by_day_hour = (
    eda_time.groupby(["day", "hour_of_day"])[LABEL_COL]
    .agg(n_txns="size", n_fraud="sum")
    .assign(fraud_rate_pct=lambda d: 100 * d["n_fraud"] / d["n_txns"])
)
volume = by_day_hour["n_txns"].unstack("day").rename(columns=lambda d: f"Day {d}")
fraud_rate = by_day_hour["fraud_rate_pct"].unstack("day").rename(columns=lambda d: f"Day {d}")

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
volume.plot(ax=axes[0], color=["steelblue", "darkorange"], lw=1.4)
axes[0].set_ylabel("All transactions")
axes[0].set_title("Transaction volume by hour of day")

fraud_rate.plot(ax=axes[1], color=["steelblue", "darkorange"], lw=1.4)
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_xlabel("Hour of day")
axes[1].set_title("Fraud rate by hour of day")
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)])

plt.tight_layout()
plt.show()

### 3.6. Analysis: Fraud over time

Fraud is **uneven by hour of day**, and the two days do not match perfectly. Most hours sit near the overall rate (~0.17%), but some slots are much hotter (about 2% fraud around Day 2, 02:00) and some have no recorded fraud. Timing therefore matters for how we evaluate models later.

---

## 4. STAGE 4: Feature preparation

Prepare unsupervised-ready inputs by retaining the PCA components and scaling `Time` and `amount`, documenting choices that avoid leaking the label into training features.



---

## 5. STAGE 5: Evaluation protocol

Define how labels are used only for evaluation, including contamination assumptions, anomaly-score ranking and any hold-out arrangement that keeps training unsupervised.



---

## 6. STAGE 6: Baseline anomaly detector

Fit a simple Isolation Forest baseline on normal-pattern features and record recall-oriented reference metrics against the evaluation labels.



---

## 7. STAGE 7: Model comparison

Compare Isolation Forest, Local Outlier Factor and One-Class SVM under the same protocol, judging each on recall, precision and alert volume.



---

## 8. STAGE 8: Threshold selection

Use precision–recall analysis to choose an operating threshold that deliberately favours recall while keeping false alarms reviewable.



---

## 9. STAGE 9: Explainability

Provide feature-attribution or similar explanations for flagged cases so manual reviewers can interpret why an alert was raised.



---

## 10. STAGE 10: Robustness, ethics and risks

Stress-test stability, document bias and fairness risks, and record GDPR-aligned constraints on automated decisions and PII use.



---

## 11. STAGE 11: Conclusions and recommendations

Summarise the strongest unsupervised approach, the cost–threshold trade-off, and practical recommendations for review workflows.



---

## 12. STAGE 12: Reproducibility appendix

Record package versions and export a requirements snapshot so the analysis can be re-run end to end.

